# Boltz-2 Negative Control Experiments — WBP5 Manuscript

**Purpose**: Calibrate the Boltz-2 confidence threshold (0.70) used in the main manuscript by running **negative control co-folding experiments** with WBP5 and 6 non-binder ligands.

**Target outcome**: Show that WBP5 + random non-binders produce Boltz-2 confidence scores well below both the 0.70 threshold AND the WBP5-pazopanib score (0.499). This validates the tier framework by anchoring the low end.

**Ligands**:
1. **Aspirin** (NSAID, MW 180, kinase-unrelated)
2. **Caffeine** (xanthine alkaloid, MW 194)
3. **Ibuprofen** (NSAID, MW 206)
4. **Glucose** (polar metabolite, MW 180)
5. **Diphenhydramine** (antihistamine, MW 255)
6. **Ascorbic acid** (vitamin C, MW 176)

> Note: pure DUD-E decoys from pazopanib are ideal but require a DUD-E lookup. We instead use 6 structurally diverse small molecules from different pharmacological classes — none of which have documented WBP5 or Hippo-pathway activity — providing a comparable non-binder negative control panel.

**Estimated runtime**: ~60 minutes on a free Colab T4 GPU (6 compounds × ~8–10 min each, plus ~5 min setup).

---

## Step 1: Enable GPU

Before running anything: **Runtime → Change runtime type → T4 GPU → Save**

Verify with the cell below.

In [ ]:
!nvidia-smi

## Step 2: Install Boltz-2

Takes ~5 minutes. The package will download model weights on first run (~2 GB).

In [1]:
!pip install boltz[cuda] --quiet
!pip install pyyaml pandas --quiet
print('Installation complete.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 266.3/266.3 kB 15.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.4/114.4 kB 11.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.2/401.2 kB 26.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 11.3 MB/s eta 0:00:00
   ━━━━

## Step 3: Define WBP5 sequence and negative-control ligands

WBP5 sequence is the 104-residue structured core from AlphaFold model AF-Q9UHQ7-F1-model_v6, matching the manuscript.

In [4]:
import os
from pathlib import Path

# WBP5 structured core (104 aa) — exactly matches the model used in the manuscript
WBP5_SEQ = 'MKSCQKMEGKPENESEPKHEEEPKPEEKPEEEEKLEEEAKAKGTFRERLIQSLQEFKEDIHNRHLSNEDMFREVDEIDEIRRVRNKLIVMRWKVNRNHPYPYLM'

assert len(WBP5_SEQ) == 104, f'Expected 104 aa, got {len(WBP5_SEQ)}'
assert WBP5_SEQ[63:74] == 'HLSNEDMFREV', 'P_0 pocket sequence mismatch'
print(f'WBP5 sequence validated: {len(WBP5_SEQ)} aa')
print(f'P_0 pocket (res 64-74): {WBP5_SEQ[63:74]}')

# Negative control ligands — all 6 have no known WBP5/Hippo-YAP1 activity
NEGATIVE_CONTROLS = {
    'aspirin':         'CC(=O)OC1=CC=CC=C1C(=O)O',
    'caffeine':        'CN1C=NC2=C1C(=O)N(C(=O)N2C)C',
    'ibuprofen':       'CC(C)CC1=CC=C(C=C1)C(C)C(=O)O',
    'glucose':         'OC[C@H]1O[C@@H](O)[C@H](O)[C@@H](O)[C@@H]1O',
    'diphenhydramine': 'CN(C)CCOC(C1=CC=CC=C1)C2=CC=CC=C2',
    'ascorbic_acid':   'OC[C@H](O)[C@H]1OC(=O)C(O)=C1O',
}

for name, smi in NEGATIVE_CONTROLS.items():
    print(f'  {name:20s} {smi}')

WBP5 sequence validated: 104 aa
P_0 pocket (res 64-74): HLSNEDMFREV
  aspirin              CC(=O)OC1=CC=CC=C1C(=O)O
  caffeine             CN1C=NC2=C1C(=O)N(C(=O)N2C)C
  ibuprofen            CC(C)CC1=CC=C(C=C1)C(C)C(=O)O
  glucose              OC[C@H]1O[C@@H](O)[C@H](O)[C@@H](O)[C@@H]1O
  diphenhydramine      CN(C)CCOC(C1=CC=CC=C1)C2=CC=CC=C2
  ascorbic_acid        OC[C@H](O)[C@H]1OC(=O)C(O)=C1O


## Step 4: Generate Boltz-2 input YAML files

In [5]:
Path('inputs').mkdir(exist_ok=True)
Path('outputs').mkdir(exist_ok=True)

for name, smiles in NEGATIVE_CONTROLS.items():
    yaml_content = f'''version: 1
sequences:
  - protein:
      id: A
      sequence: {WBP5_SEQ}
  - ligand:
      id: B
      smiles: "{smiles}"
'''
    with open(f'inputs/wbp5_{name}.yaml', 'w') as f:
        f.write(yaml_content)
    print(f'Created inputs/wbp5_{name}.yaml')

Created inputs/wbp5_aspirin.yaml
Created inputs/wbp5_caffeine.yaml
Created inputs/wbp5_ibuprofen.yaml
Created inputs/wbp5_glucose.yaml
Created inputs/wbp5_diphenhydramine.yaml
Created inputs/wbp5_ascorbic_acid.yaml


## Step 5: Run Boltz-2 predictions

This is the long step. ~8–10 minutes per compound × 6 = ~60 min total. The first compound takes extra time (~12 min) because Boltz-2 downloads model weights.

> If Colab disconnects mid-run, just re-run this cell — completed runs are skipped.

In [6]:
import subprocess
import time

for name in NEGATIVE_CONTROLS.keys():
    output_dir = f'outputs/{name}'
    confidence_file_pattern = f'{output_dir}/boltz_results_wbp5_{name}/predictions/wbp5_{name}/confidence_wbp5_{name}_model_0.json'

    if Path(confidence_file_pattern).exists():
        print(f'✓ {name} already done, skipping')
        continue

    print(f'\n=== Running Boltz-2 for WBP5 + {name} ===')
    start = time.time()
    result = subprocess.run(
        ['boltz', 'predict', f'inputs/wbp5_{name}.yaml',
         '--out_dir', output_dir,
         '--use_msa_server',
         '--output_format', 'pdb'],
        capture_output=True, text=True
    )
    elapsed = time.time() - start
    if result.returncode != 0:
        print(f'ERROR for {name} after {elapsed:.0f}s:')
        print(result.stderr[-2000:])
    else:
        print(f'✓ Completed in {elapsed:.0f}s')

print('\nAll predictions complete!')


=== Running Boltz-2 for WBP5 + aspirin ===
✓ Completed in 138s

=== Running Boltz-2 for WBP5 + caffeine ===
✓ Completed in 86s

=== Running Boltz-2 for WBP5 + ibuprofen ===
✓ Completed in 84s

=== Running Boltz-2 for WBP5 + glucose ===
✓ Completed in 82s

=== Running Boltz-2 for WBP5 + diphenhydramine ===
✓ Completed in 81s

=== Running Boltz-2 for WBP5 + ascorbic_acid ===
✓ Completed in 76s

All predictions complete!


## Step 6: Parse results and generate summary table

In [8]:
import json
import pandas as pd
import glob

results = []
for name in NEGATIVE_CONTROLS.keys():
    # Boltz output path may vary — search for the confidence file
    matches = glob.glob(f'outputs/{name}/**/confidence_wbp5_{name}_model_0.json', recursive=True)
    if not matches:
        print(f'⚠ No output found for {name}')
        results.append({'ligand': name, 'status': 'missing'})
        continue

    with open(matches[0]) as f:
        data = json.load(f)

    results.append({
        'ligand': name,
        'confidence_score': data.get('confidence_score', None),
        'ptm': data.get('ptm', None),
        'iptm': data.get('iptm', None),
        'ligand_iptm': data.get('ligand_iptm', None),
        'complex_plddt': data.get('complex_plddt', None),
        'tier': 'exploratory' if data.get('confidence_score', 0) >= 0.50 else 'below-exploratory'
    })

df = pd.DataFrame(results)

# Add the published reference rows for context
reference_rows = pd.DataFrame([
    {'ligand': 'EGFR–erlotinib (positive ctrl)', 'confidence_score': 0.788, 'iptm': None,
     'ligand_iptm': 0.981, 'tier': 'high-confidence'},
    {'ligand': 'WBP5–pazopanib (focal)',         'confidence_score': 0.499, 'iptm': None,
     'ligand_iptm': 0.478, 'tier': 'exploratory'},
])

# Display full comparison
print('\n==== BOLTZ-2 CO-FOLDING RESULTS ====\n')
full_df = pd.concat([reference_rows, df], ignore_index=True)
print(full_df.to_string(index=False))

# Save to CSV
full_df.to_csv('boltz2_negative_control_results.csv', index=False)
print('\nSaved: boltz2_negative_control_results.csv')

# Compute summary statistics for the negative controls
neg_conf = [r['confidence_score'] for r in results if r.get('confidence_score') is not None]
neg_ipTM = [r['ligand_iptm'] for r in results if r.get('ligand_iptm') is not None]
if neg_conf:
    print(f'\nNegative control summary (n={len(neg_conf)}):')
    print(f'  Confidence score: mean={sum(neg_conf)/len(neg_conf):.3f}, min={min(neg_conf):.3f}, max={max(neg_conf):.3f}')
    print(f'  Ligand ipTM:      mean={sum(neg_ipTM)/len(neg_ipTM):.3f}, min={min(neg_ipTM):.3f}, max={max(neg_ipTM):.3f}')

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [10]:
import json
import glob
import csv

NEGATIVE_CONTROLS_LIST = ['aspirin', 'caffeine', 'ibuprofen', 'glucose', 'diphenhydramine', 'ascorbic_acid']

results = []
print("="*75)
print(f"{'Ligand':<30} {'Conf':>8} {'ipTM':>8} {'LigIpTM':>10} {'cPLDDT':>8}")
print("="*75)

# Reference rows first
print(f"{'EGFR-erlotinib (pos ctrl)':<30} {0.788:>8.3f} {'--':>8} {0.981:>10.3f} {'--':>8}")
print(f"{'WBP5-pazopanib (focal)':<30} {0.499:>8.3f} {'--':>8} {0.478:>10.3f} {'--':>8}")
print("-"*75)

for name in NEGATIVE_CONTROLS_LIST:
    matches = glob.glob(f'outputs/{name}/**/confidence_wbp5_{name}_model_0.json', recursive=True)
    if not matches:
        print(f"{name:<30} {'MISSING':>8}")
        results.append({'ligand': name, 'status': 'missing'})
        continue
    with open(matches[0]) as f:
        data = json.load(f)
    conf = data.get('confidence_score')
    ptm = data.get('ptm')
    iptm = data.get('iptm')
    lig_iptm = data.get('ligand_iptm')
    cplddt = data.get('complex_plddt')
    print(f"{name:<30} {conf:>8.3f} {iptm if iptm else 0:>8.3f} {lig_iptm if lig_iptm else 0:>10.3f} {cplddt if cplddt else 0:>8.2f}")
    results.append({
        'ligand': name, 'confidence_score': conf, 'ptm': ptm, 'iptm': iptm,
        'ligand_iptm': lig_iptm, 'complex_plddt': cplddt,
        'tier': 'exploratory' if conf >= 0.50 else 'below-exploratory'
    })

print("="*75)

# Save CSV without pandas
with open('boltz2_negative_control_results.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['ligand', 'confidence_score', 'ptm', 'iptm', 'ligand_iptm', 'complex_plddt', 'tier'])
    writer.writerow(['EGFR-erlotinib (pos ctrl)', 0.788, '', '', 0.981, '', 'high-confidence'])
    writer.writerow(['WBP5-pazopanib (focal)', 0.499, '', '', 0.478, '', 'exploratory'])
    for r in results:
        if r.get('status') == 'missing':
            writer.writerow([r['ligand'], '', '', '', '', '', 'MISSING'])
        else:
            writer.writerow([r['ligand'], r['confidence_score'], r.get('ptm', ''),
                           r.get('iptm', ''), r.get('ligand_iptm', ''),
                           r.get('complex_plddt', ''), r['tier']])
print("\n✅ Saved: boltz2_negative_control_results.csv")

# Summary statistics
neg_conf = [r['confidence_score'] for r in results if r.get('confidence_score') is not None]
neg_ipTM = [r['ligand_iptm'] for r in results if r.get('ligand_iptm') is not None]
if neg_conf:
    print(f"\n=== Negative control summary (n={len(neg_conf)}) ===")
    print(f"  Confidence: mean={sum(neg_conf)/len(neg_conf):.3f}, min={min(neg_conf):.3f}, max={max(neg_conf):.3f}")
    print(f"  Lig ipTM:   mean={sum(neg_ipTM)/len(neg_ipTM):.3f}, min={min(neg_ipTM):.3f}, max={max(neg_ipTM):.3f}")

Ligand                             Conf     ipTM    LigIpTM   cPLDDT
EGFR-erlotinib (pos ctrl)         0.788       --      0.981       --
WBP5-pazopanib (focal)            0.499       --      0.478       --
---------------------------------------------------------------------------
aspirin                           0.667    0.840      0.840     0.62
caffeine                          0.707    0.879      0.879     0.66
ibuprofen                         0.715    0.943      0.943     0.66
glucose                           0.664    0.906      0.906     0.60
diphenhydramine                   0.657    0.795      0.795     0.62
ascorbic_acid                     0.684    0.907      0.907     0.63

✅ Saved: boltz2_negative_control_results.csv

=== Negative control summary (n=6) ===
  Confidence: mean=0.683, min=0.657, max=0.715
  Lig ipTM:   mean=0.878, min=0.795, max=0.943


## Step 7: Download results

Right-click on `boltz2_negative_control_results.csv` in the Colab file panel and select **Download**.

Then send the CSV file back to Claude, which will:
1. Parse the final confidence/ipTM values
2. Update **Table 5** in the manuscript (add 6 new negative-control rows)
3. Update **Results §3.7** narrative with the negative-control data
4. Update **Response to Reviewers A5** to cite the actual numbers
5. Update **Figure 4 / Graphical Abstract** if needed to show the 3-tier spectrum (high-confidence / exploratory / below-exploratory)

---

## Troubleshooting

- **Out of GPU memory**: Runtime → Disconnect and delete runtime → reconnect with fresh T4
- **Colab disconnects**: Just re-run Step 5 — completed compounds skip automatically
- **Install fails**: Ensure GPU runtime is selected (Step 1 check)
- **Confidence JSON not found**: Check `outputs/{ligand}/` directory manually for the actual file path